# 10 — Finding the right scorer: architecture and hyperparameters

The published scorer has **304,049 parameters fitted on 206 windows** — roughly 1,170
parameters per training sample. That is not a model that can be expected to behave
reproducibly, and it did not: rerunning identical code moved its window AUC from 0.910
to 0.948.

This notebook searches for a scorer matched to the amount of data that actually
exists, evaluated by **5-fold cross-validation grouped by recording** over all 156
sessions. Out-of-fold pooling means every recording is scored exactly once by a model
that never saw it, giving ~520 evaluation windows instead of the 167 a single held-out
split provided.

## Selection rule, fixed before looking at any result

Written down first, deliberately, because choosing the criterion after seeing the
numbers is how selection bias gets in — and I made exactly that mistake earlier in this
project.

1. **Primary:** session-level AUC. This is what the simulation consumes — the scorer
   averages over a chunk's windows, so chunk/session level is the operating point,
   not window level.
2. **Among configurations within 1 standard error of the best session AUC**, prefer the
   lowest post-temperature-scaling ECE. Calibration is not cosmetic here: the
   multi-threshold baseline maps the score directly onto a threat level with
   `int(score * 10)`.
3. **Among those**, prefer fewer parameters. Parsimony is the whole point of the
   exercise.

**One amendment, made before running anything and for a stated reason.** Notebook 09
found that a generic model loses 0.126 session AUC and sees its ECE rise from 0.055 to
0.220 when tested on a collection sitting it never saw. That is a property of the
*data* — three participants confounded with two campaigns — not of any particular
architecture, so acting on it is not selection bias. Stage C therefore also evaluates
each finalist leave-one-campaign-out, and a configuration that collapses there is
rejected even if it tops the cross-validated ranking. A scorer that only works on the
sitting it was trained on is not usable for the RL work.

In [1]:
import json, os, sys, time
from pathlib import Path
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
sys.path.insert(0, str(Path.cwd()))

import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

import expkit
from expkit.partition import index_sessions
from expkit.paths import RESULTS
from expkit.scorer_search import ArchSpec, PUBLISHED, cross_validate, save, to_frame

OUT = RESULTS
REFS = index_sessions()
EPOCHS, PATIENCE, N_SPLITS = 90, 12, 5
print(f"{len(REFS)} recordings, {N_SPLITS}-fold grouped by recording")

156 recordings, 5-fold grouped by recording


## 10.1 Stage A — capacity and input representation

Six architectures crossed with four input representations. The published model is
included so every later comparison has it as a reference point rather than a memory.

The representations:

| name | features | what it tests |
|---|---|---|
| `xy` | x, y | the published input |
| `xy_dt` | x, y, Δt | does timing help at all? |
| `dxdy` | Δx, Δy | translation-invariant, no timing — the control |
| `dxdy_dt` | Δx, Δy, Δt | translation-invariant plus timing |
| `kinematic` | Δx, Δy, Δt, speed, accel | the features the manuscript claims to use |

In [2]:
ARCHS = [
    ArchSpec("lstm16",      (16,),      (8,)),
    ArchSpec("lstm16_8",    (16, 8),    (16,)),
    ArchSpec("lstm32",      (32,),      (32,)),
    ArchSpec("lstm32_16",   (32, 16),   (32,)),
    ArchSpec("lstm64_32",   (64, 32),   (64,)),
    PUBLISHED,
]
REPS = ["xy", "xy_dt", "dxdy", "dxdy_dt", "kinematic"]

results_a, t0 = [], time.time()
for arch in ARCHS:
    for rep in REPS:
        r = cross_validate(REFS, arch, rep, n_splits=N_SPLITS, group="session",
                           seed=0, epochs=EPOCHS, patience=PATIENCE)
        results_a.append(r)
        print(f"  {arch.name:12s} {rep:10s} params={r.params:>7,}  "
              f"winAUC={r.window_auc:.3f} sessAUC={r.session_auc:.3f} "
              f"ECE={r.ece:.3f}->{r.ece_after_T:.3f}  [{(time.time()-t0)/60:.0f}m]")
save(results_a, "stage_a")
A = to_frame(results_a)
A.to_csv(OUT / "search_stage_a.csv", index=False)

  lstm16       xy         params=  1,361  winAUC=0.881 sessAUC=0.908 ECE=0.048->0.044  [3m]


  lstm16       xy_dt      params=  1,425  winAUC=0.893 sessAUC=0.921 ECE=0.046->0.049  [4m]


  lstm16       dxdy       params=  1,361  winAUC=0.809 sessAUC=0.894 ECE=0.064->0.064  [5m]


  lstm16       dxdy_dt    params=  1,425  winAUC=0.931 sessAUC=0.951 ECE=0.035->0.034  [6m]


  lstm16       kinematic  params=  1,553  winAUC=0.935 sessAUC=0.960 ECE=0.036->0.041  [7m]


...

  published    xy         params=304,049  winAUC=0.861 sessAUC=0.904 ECE=0.067->0.065  [52m]


  published    xy_dt      params=304,849  winAUC=0.855 sessAUC=0.889 ECE=0.043->0.043  [59m]


  published    dxdy       params=304,049  winAUC=0.879 sessAUC=0.926 ECE=0.030->0.035  [70m]


  published    dxdy_dt    params=304,849  winAUC=0.854 sessAUC=0.919 ECE=0.044->0.058  [78m]


  published    kinematic  params=306,449  winAUC=0.937 sessAUC=0.948 ECE=0.056->0.064  [83m]


In [3]:
print("\nsession AUC by architecture x representation:")
print(A.pivot_table(index="spec", columns="representation",
                    values="session_auc").round(3).to_string())
print("\npost-scaling ECE:")
print(A.pivot_table(index="spec", columns="representation",
                    values="ece_after_T").round(3).to_string())
print("\nparameters per training sample (~250 windows):")
print((A.groupby("spec").params.first() / 250).round(1).to_string())


session AUC by architecture x representation:
representation   dxdy  dxdy_dt  kinematic     xy  xy_dt
spec                                                   
lstm16          0.894    0.951      0.960  0.908  0.921
lstm16_8        0.968    0.967      0.947  0.906  0.910
lstm32          0.971    0.980      0.965  0.941  0.920
lstm32_16       0.956    0.933      0.954  0.908  0.909
lstm64_32       0.962    0.969      0.969  0.884  0.889
published       0.926    0.919      0.948  0.904  0.889

post-scaling ECE:
representation   dxdy  dxdy_dt  kinematic     xy  xy_dt
spec                                                   
lstm16          0.064    0.034      0.041  0.044  0.049
lstm16_8        0.021    0.056      0.050  0.034  0.068
lstm32          0.028    0.025      0.036  0.055  0.034
lstm32_16       0.028    0.030      0.026  0.056  0.036
lstm64_32       0.055    0.040      0.027  0.046  0.026
published       0.035    0.058      0.064  0.065  0.043

parameters per training sample (~250 

## 10.2 Stage B — learning rate and regularisation

The three architectures with the best mean session AUC across representations, each
with its own best representation, swept over learning rate and dropout.

In [4]:
rank_arch = A.groupby("spec").session_auc.mean().sort_values(ascending=False)
top_specs = [s for s in rank_arch.index[:3]]
best_rep = {s: A[A.spec == s].sort_values("session_auc", ascending=False)
                .representation.iloc[0] for s in top_specs}
print("carried into stage B:", {s: best_rep[s] for s in top_specs})

BY_NAME = {a.name: a for a in ARCHS}
results_b = []
t0 = time.time()
for s in top_specs:
    base = BY_NAME[s]
    for lr in (3e-4, 1e-3, 3e-3):
        for drop in (0.0, 0.25):
            import dataclasses
            spec = dataclasses.replace(base, name=f"{s}_lr{lr:g}_d{drop:g}",
                                       lr=lr, dropout=drop)
            r = cross_validate(REFS, spec, best_rep[s], n_splits=N_SPLITS,
                               group="session", seed=0, epochs=EPOCHS, patience=PATIENCE)
            results_b.append(r)
            print(f"  {spec.name:26s} {best_rep[s]:10s} "
                  f"sessAUC={r.session_auc:.3f} ECE={r.ece_after_T:.3f} "
                  f"[{(time.time()-t0)/60:.0f}m]")
save(results_b, "stage_b")
B = to_frame(results_b)
B.to_csv(OUT / "search_stage_b.csv", index=False)
print("\n", B[["spec", "representation", "params", "session_auc", "window_auc",
               "ece", "ece_after_T", "temperature"]].round(3)
        .sort_values("session_auc", ascending=False).to_string(index=False))

carried into stage B: {'lstm32': 'dxdy_dt', 'lstm16_8': 'dxdy', 'lstm64_32': 'kinematic'}


  lstm32_lr0.0003_d0         dxdy_dt    sessAUC=0.976 ECE=0.022 [2m]


  lstm32_lr0.0003_d0.25      dxdy_dt    sessAUC=0.962 ECE=0.072 [3m]


  lstm32_lr0.001_d0          dxdy_dt    sessAUC=0.980 ECE=0.025 [6m]


  lstm32_lr0.001_d0.25       dxdy_dt    sessAUC=0.963 ECE=0.047 [8m]


...

  lstm64_32_lr0.0003_d0.25   kinematic  sessAUC=0.949 ECE=0.049 [38m]


  lstm64_32_lr0.001_d0       kinematic  sessAUC=0.969 ECE=0.027 [41m]


  lstm64_32_lr0.001_d0.25    kinematic  sessAUC=0.968 ECE=0.051 [45m]


  lstm64_32_lr0.003_d0       kinematic  sessAUC=0.949 ECE=0.050 [47m]


  lstm64_32_lr0.003_d0.25    kinematic  sessAUC=0.965 ECE=0.047 [50m]

                     spec representation  params  session_auc  window_auc   ece  ece_after_T  temperature
       lstm32_lr0.001_d0        dxdy_dt    5697        0.980       0.963 0.025        0.025        1.015
      lstm32_lr0.0003_d0        dxdy_dt    5697        0.976       0.959 0.024        0.022        1.060
 lstm16_8_lr0.0003_d0.25           dxdy    2177        0.976       0.952 0.034        0.022        0.852
  lstm16_8_lr0.001_d0.25           dxdy    2177        0.975       0.956 0.037        0.025        0.840
    lstm64_32_lr0.001_d0      kinematic   32513        0.969       0.968 0.033        0.027        1.268
     lstm16_8_lr0.001_d0           dxdy    2177        0.968       0.957 0.022        0.021        0.948
 lstm64_32_lr0.001_d0.25      kinematic   32513        0.968       0.954 0.052        0.051        1.323
    lstm16_8_lr0.0003_d0           dxdy    2177        0.968       0.950 0.032        0.

## 10.3 Stage C — is the winner stable across seeds?

The single most important check, given that this project has already been bitten by
seed instability once. A configuration that only wins on seed 0 has not won.

In [5]:
ALL = pd.concat([A, B], ignore_index=True)
finalists = (ALL.sort_values("session_auc", ascending=False)
                .drop_duplicates("spec").head(4))
print("finalists:\n", finalists[["spec", "representation", "params", "session_auc",
                                 "ece_after_T"]].round(3).to_string(index=False))

SPECS_BY_NAME = {a.name: a for a in ARCHS}
for r in results_b:
    import dataclasses
    base_name = r.spec.split("_lr")[0]
    lr = float(r.spec.split("_lr")[1].split("_d")[0])
    drop = float(r.spec.split("_d")[1])
    SPECS_BY_NAME[r.spec] = dataclasses.replace(SPECS_BY_NAME[base_name],
                                                name=r.spec, lr=lr, dropout=drop)

results_c, t0 = [], time.time()
for _, row in finalists.iterrows():
    for seed in (0, 1, 2):
        r = cross_validate(REFS, SPECS_BY_NAME[row.spec], row.representation,
                           n_splits=N_SPLITS, group="session", seed=seed,
                           epochs=EPOCHS, patience=PATIENCE)
        results_c.append(r)
    sub = [x for x in results_c if x.spec == row.spec]
    print(f"  {row.spec:26s} sessAUC={np.mean([x.session_auc for x in sub]):.3f}"
          f" +/-{np.std([x.session_auc for x in sub]):.3f}  "
          f"ECE={np.mean([x.ece_after_T for x in sub]):.3f}  [{(time.time()-t0)/60:.0f}m]")
save(results_c, "stage_c")
C = to_frame(results_c)
C.to_csv(OUT / "search_stage_c.csv", index=False)

stab = C.groupby(["spec", "representation"]).agg(
    params=("params", "first"),
    sess_auc=("session_auc", "mean"), sess_auc_sd=("session_auc", "std"),
    win_auc=("window_auc", "mean"), win_auc_sd=("window_auc", "std"),
    ece=("ece", "mean"), ece_T=("ece_after_T", "mean"), ece_T_sd=("ece_after_T", "std"),
    T=("temperature", "mean")).reset_index().sort_values("sess_auc", ascending=False)
print("\n", stab.round(3).to_string(index=False))

finalists:
                    spec representation  params  session_auc  ece_after_T
                 lstm32        dxdy_dt    5697        0.980        0.025
      lstm32_lr0.001_d0        dxdy_dt    5697        0.980        0.025
lstm16_8_lr0.0003_d0.25           dxdy    2177        0.976        0.022
     lstm32_lr0.0003_d0        dxdy_dt    5697        0.976        0.022


  lstm32                     sessAUC=0.973 +/-0.009  ECE=0.035  [7m]


  lstm32_lr0.001_d0          sessAUC=0.973 +/-0.009  ECE=0.035  [15m]


  lstm16_8_lr0.0003_d0.25    sessAUC=0.914 +/-0.073  ECE=0.054  [24m]


  lstm32_lr0.0003_d0         sessAUC=0.946 +/-0.044  ECE=0.039  [31m]

                    spec representation  params  sess_auc  sess_auc_sd  win_auc  win_auc_sd   ece  ece_T  ece_T_sd     T
                 lstm32        dxdy_dt    5697     0.973        0.011    0.949       0.020 0.031  0.035     0.014 0.947
      lstm32_lr0.001_d0        dxdy_dt    5697     0.973        0.011    0.949       0.020 0.031  0.035     0.014 0.947
     lstm32_lr0.0003_d0        dxdy_dt    5697     0.946        0.054    0.929       0.046 0.032  0.039     0.016 0.873
lstm16_8_lr0.0003_d0.25           dxdy    2177     0.914        0.089    0.861       0.112 0.055  0.054     0.046 0.795


### Robustness gate — leave-one-campaign-out

Each finalist re-fitted with a whole collection sitting held out. This is the closest
proxy available for an unseen participant and setup.

In [6]:
loco = []
for _, row in stab.iterrows():
    r = cross_validate(REFS, SPECS_BY_NAME[row.spec], row.representation,
                       n_splits=2, group="campaign", seed=0,
                       epochs=EPOCHS, patience=PATIENCE)
    loco.append({"spec": row.spec, "representation": row.representation,
                 "loco_win_auc": r.window_auc, "loco_sess_auc": r.session_auc,
                 "loco_ece": r.ece, "loco_ece_T": r.ece_after_T})
    print(f"  {row.spec:26s} LOCO sessAUC={r.session_auc:.3f} "
          f"ECE={r.ece:.3f}->{r.ece_after_T:.3f}")
L = pd.DataFrame(loco)
L.to_csv(OUT / "search_loco.csv", index=False)
stab = stab.merge(L, on=["spec", "representation"], how="left")
print("\ncross-validated vs held-out-campaign, same models:")
print(stab[["spec", "representation", "sess_auc", "loco_sess_auc",
            "ece_T", "loco_ece_T"]].round(3).to_string(index=False))
stab["optimism"] = (stab.sess_auc - stab.loco_sess_auc).round(3)

  lstm32                     LOCO sessAUC=0.696 ECE=0.211->0.214


  lstm32_lr0.001_d0          LOCO sessAUC=0.696 ECE=0.211->0.214


  lstm32_lr0.0003_d0         LOCO sessAUC=0.686 ECE=0.177->0.168


  lstm16_8_lr0.0003_d0.25    LOCO sessAUC=0.792 ECE=0.124->0.133

cross-validated vs held-out-campaign, same models:
                   spec representation  sess_auc  loco_sess_auc  ece_T  loco_ece_T
                 lstm32        dxdy_dt     0.973          0.696  0.035       0.214
      lstm32_lr0.001_d0        dxdy_dt     0.973          0.696  0.035       0.214
     lstm32_lr0.0003_d0        dxdy_dt     0.946          0.686  0.039       0.168
lstm16_8_lr0.0003_d0.25           dxdy     0.914          0.792  0.054       0.133


## 10.4 Applying the selection rule

In [7]:
# The gate is applied to ALL finalists FIRST. Applying it inside the 1-SE band, as an
# earlier version did, meant that when every member of the band failed the gate the
# fallback quietly reinstated them -- the gate did nothing in exactly the case it was
# written for.
gate = stab.loco_sess_auc.max() - 0.05
survivors = stab[stab.loco_sess_auc >= gate].copy()
rejected = stab[stab.loco_sess_auc < gate]
print(f"robustness gate: LOCO session AUC >= {gate:.3f} "
      f"(best LOCO {stab.loco_sess_auc.max():.3f} minus 0.05)")
if len(rejected):
    print("\nrejected -- strong cross-validated but collapses on an unseen sitting:")
    print(rejected[["spec", "representation", "sess_auc",
                    "loco_sess_auc"]].round(3).to_string(index=False))

best = survivors.iloc[0]
se = best.sess_auc_sd / np.sqrt(3)
within = survivors[survivors.sess_auc >= best.sess_auc - se].copy()
print(f"\nbest surviving session AUC {best.sess_auc:.3f}, 1 SE = {se:.3f}  ->  "
      f"{len(within)} configuration(s) within it")
within = within.sort_values(["ece_T", "params"])
print(within[["spec", "representation", "params", "sess_auc", "sess_auc_sd",
              "ece_T", "loco_sess_auc", "optimism"]].round(3).to_string(index=False))
choice = within.iloc[0]
print(f"\nSELECTED: {choice.spec} on {choice.representation}  "
      f"({int(choice.params):,} params, session AUC {choice.sess_auc:.3f} "
      f"+/-{choice.sess_auc_sd:.3f}, ECE after scaling {choice.ece_T:.3f})")

pub_row = stab[stab.spec == "published"]
if len(pub_row) == 0:
    pub_row = ALL[ALL.spec == "published"].sort_values("session_auc",
                                                       ascending=False).head(1)
print(f"\nfor reference, published architecture: "
      f"{int(pub_row.iloc[0]['params']):,} params, "
      f"session AUC {pub_row.iloc[0].get('sess_auc', pub_row.iloc[0].get('session_auc')):.3f}")

json.dump({"spec": choice.spec, "representation": choice.representation,
           "params": int(choice.params), "session_auc": float(choice.sess_auc),
           "ece_after_T": float(choice.ece_T),
           "loco_session_auc": float(choice.loco_sess_auc),
           "loco_ece_after_T": float(choice.loco_ece_T),
           "rule": "max session AUC; within 1 SE apply the leave-one-campaign-out "
                   "robustness gate, then prefer lowest post-scaling ECE, then "
                   "fewest parameters"},
          open(OUT / "scorer_arch_choice.json", "w"), indent=2)

robustness gate: LOCO session AUC >= 0.742 (best LOCO 0.792 minus 0.05)

rejected -- strong cross-validated but collapses on an unseen sitting:
              spec representation  sess_auc  loco_sess_auc
            lstm32        dxdy_dt     0.973          0.696
 lstm32_lr0.001_d0        dxdy_dt     0.973          0.696
lstm32_lr0.0003_d0        dxdy_dt     0.946          0.686

best surviving session AUC 0.914, 1 SE = 0.051  ->  1 configuration(s) within it
                   spec representation  params  sess_auc  sess_auc_sd  ece_T  loco_sess_auc  optimism
lstm16_8_lr0.0003_d0.25           dxdy    2177     0.914        0.089  0.054          0.792     0.122

SELECTED: lstm16_8_lr0.0003_d0.25 on dxdy  (2,177 params, session AUC 0.914 +/-0.089, ECE after scaling 0.054)

for reference, published architecture: 306,449 params, session AUC 0.948


## 10.5 Does timing actually help? — the question notebook 02 could not answer

Notebook 02 compared representations on a single 52-session holdout and every interval
straddled zero. With 5-fold cross-validation the evaluation set is ~3x larger, so the
intervals should be ~1.8x tighter. This is where we find out whether that is enough.

In [8]:
rep_cmp = (A.groupby("representation")
             .agg(mean_sess_auc=("session_auc", "mean"),
                  best_sess_auc=("session_auc", "max"),
                  mean_ece_T=("ece_after_T", "mean")).round(3))
print("averaged over all six architectures:\n", rep_cmp.to_string())
print("\nper architecture, session AUC of each representation minus `xy`:")
delta = A.pivot_table(index="spec", columns="representation", values="session_auc")
print((delta.sub(delta["xy"], axis=0)).round(3).to_string())
print("""
Read the sign consistency, not the size. If a representation beats `xy` for every
architecture, that is evidence even when each individual gap is small; if the sign flips
between architectures, the differences are noise.""")

averaged over all six architectures:
                 mean_sess_auc  best_sess_auc  mean_ece_T
representation                                          
dxdy                    0.946          0.971       0.038
dxdy_dt                 0.953          0.980       0.041
kinematic               0.957          0.969       0.041
xy                      0.909          0.941       0.050
xy_dt                   0.907          0.921       0.043

per architecture, session AUC of each representation minus `xy`:
representation   dxdy  dxdy_dt  kinematic   xy  xy_dt
spec                                                 
lstm16         -0.014    0.044      0.052  0.0  0.014
lstm16_8        0.061    0.061      0.041  0.0  0.004
lstm32          0.030    0.039      0.024  0.0 -0.020
lstm32_16       0.048    0.025      0.047  0.0  0.002
lstm64_32       0.078    0.084      0.084  0.0  0.005
published       0.021    0.014      0.043  0.0 -0.015

Read the sign consistency, not the size. If a representation bea

## 10.6 Figures

In [9]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))

for rep in REPS:
    s = A[A.representation == rep].sort_values("params")
    ax[0].plot(s.params, s.session_auc, "o-", ms=5, label=rep)
ax[0].set_xscale("log"); ax[0].set_xlabel("parameters"); ax[0].set_ylabel("session AUC")
ax[0].set_title("Capacity vs performance"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

ax[1].scatter(A.session_auc, A.ece_after_T, s=30, c=np.log10(A.params), cmap="viridis")
for _, r in A.iterrows():
    if r.params > 100000 or r.session_auc > A.session_auc.quantile(.9):
        ax[1].annotate(r.spec[:9], (r.session_auc, r.ece_after_T), fontsize=6)
ax[1].set_xlabel("session AUC"); ax[1].set_ylabel("ECE after temperature scaling")
ax[1].set_title("Discrimination vs calibration (colour = log params)")
ax[1].grid(alpha=.3)

st = stab.head(6)
x = np.arange(len(st))
ax[2].bar(x, st.sess_auc, yerr=st.sess_auc_sd, capsize=3)
ax[2].set_xticks(x); ax[2].set_xticklabels(st.spec, rotation=30, ha="right", fontsize=6.5)
ax[2].set_ylim(0.5, 1.0); ax[2].set_ylabel("session AUC (3 seeds)")
ax[2].set_title("Stability of the finalists"); ax[2].grid(alpha=.3, axis="y")

plt.tight_layout(); plt.savefig(OUT / "scorer_search.pdf", dpi=300)
plt.savefig(OUT / "scorer_search.png", dpi=110); plt.close()

with pd.ExcelWriter(OUT / "scorer_search.xlsx", engine="openpyxl") as xl:
    A.to_excel(xl, sheet_name="Stage A arch x rep", index=False)
    B.to_excel(xl, sheet_name="Stage B lr x dropout", index=False)
    stab.to_excel(xl, sheet_name="Stage C seeds", index=False)
    rep_cmp.to_excel(xl, sheet_name="Representation")
print("Notebook 10 complete.")

Notebook 10 complete.
